In [90]:
from dataclasses import dataclass,field
import os.path
from os import listdir
import sys
from enum import Enum
import importlib
import json
import jsons
import math
import requests
import datetime
import gzip

from collections import OrderedDict

#Change input_directory to Elite Insight log directory
input_directory = 'C:\\GW2Logs\\Output\\'
files = listdir(input_directory)
sorted_files = sorted(files)
test_text=""
players_running_healing_addon=[]
Regen_Healing={}

for filename in sorted_files:
    
    file_start, file_extension = os.path.splitext(filename)
    # skip files of incorrect filetype
    if file_extension not in ['.json', '.gz']:
        continue
    #if filename not in ['TW5_top_stats_202507151431.json']:
        #continue
    file_path = "".join((input_directory,"/",filename))

    if file_extension == '.gz':
        with gzip.open(file_path, mode="r") as f:
            json_data = json.loads(f.read().decode('utf-8'))
    else:
        json_datafile = open(file_path, encoding='utf-8')
        json_data = json.load(json_datafile)    

    print(f"Processing file: {file_start}")
    
    if 'usedExtensions' not in json_data:
        players_running_healing_addon = []
    else:
        extensions = json_data['usedExtensions']
        for extension in extensions:
            if extension['name'] == "Healing Stats":
                #players_running_healing_addon = extension['runningExtension']
                for healer_name in extension['runningExtension']:
                    if healer_name not in players_running_healing_addon:
                        players_running_healing_addon.append(healer_name)
                
    players = json_data['players']                
    
    
    for player in players:
        if player['name'] not in players_running_healing_addon:
            continue
        print(f"processing player: {player['name']}")
        Healing_dist = player['extHealingStats']['alliedHealingDist']
        for index, target in enumerate(Healing_dist):
            for skilllist in [target][0]:
                for skill in skilllist:
                    print(skill["id"])
                    if skill['id'] == 718:
                        print('Regen Skill Found, collect healing')
                        target_name = players[index]['name']
                        target_heal = skill['totalHealing']
                        target_hits = skill['hits']
    
                        if target_name not in Regen_Healing:
                            Regen_Healing[target_name] = {}
 
                        if player['name'] not in Regen_Healing[target_name]:
                            Regen_Healing[target_name][player['name']] = {}
    
                        Regen_Healing[target_name][player['name']]['healing'] = Regen_Healing[target_name][player['name']].get('healing',0)+target_heal
                        Regen_Healing[target_name][player['name']]['hits'] = Regen_Healing[target_name][player['name']].get('hits',0)+target_hits
                            
print('---=====Complete=====---')
print(test_text)


Processing file: 20250825-212103_detailed_wvw_kill.json
processing player: Mac Otterette
5587
718
Regen Skill Found, collect healing
5681
12836
5587
5549
5587
5549
718
Regen Skill Found, collect healing
718
Regen Skill Found, collect healing
5587
5587
5549
12836
12836
5549
5549
5549
12836
5549
5549
5549
5549
12836
5681
5549
5549
processing player: Ms Elitia
9140
9140
46298
9140
9140
9140
9140
9140
9265
9265
9140
9140
9265
9265
processing player: Praetorian Nades
59601
processing player: Amalgam Cael
59601
processing player: Drevarr Moonwillow
30564
71882
71882
71882
71882
718
Regen Skill Found, collect healing
49090
71882
718
Regen Skill Found, collect healing
71882
71882
718
Regen Skill Found, collect healing
71882
71882
30564
71882
30564
71882
20462
71882
71882
20462
71882
71882
30564
71882
20462
71882
30564
processing player: Silvyrs
9140
9140
9265
9140
9140
46298
9140
9265
9140
9140
9140
9140
9140
9265
9265
9140
9265
9140
9140
9265
9265
9140
9265
9140
9265
9265
processing player: A

In [91]:
players_running_healing_addon

['Drevarr Moonwillow',
 'Serafina Eloise',
 'Silvyrs',
 'Mac Otterette',
 'Praetorian Nades',
 'Reppalskay',
 'Chocolate Berries',
 'Amalgam Cael',
 'Ms Elitia',
 'Upper D Amalgam',
 'Adapted In The Mist',
 'Corrupted Deman',
 'Hanbee Ele',
 'Mistwarden Cael',
 'Komyni',
 'Hang L',
 'Kanta Kahn',
 'Arcana In The Mist',
 'Relexis',
 'Elemental Ká Ôs',
 'Gandalffa Beta']

In [76]:
Regen_Healing['Drevarr Moonwillow']

{'Drevarr Moonwillow': {'healing': 25536, 'hits': 70},
 'Reppalskay': {'healing': 1352, 'hits': 2}}

In [97]:
header = "|Heal Target |"
for name in players_running_healing_addon:
    header+=f" !{name[:10]}|"
header+="h"
print(header)
line=""
for player in Regen_Healing:
    line+=f"|{player} |"
    for healer in players_running_healing_addon:
        if healer in Regen_Healing[player]:
            line+=f" {Regen_Healing[player][healer]['healing']:,.0f}|"
        else:
            line+=f" |"
    line+="\n"
print(line)

|Heal Target | !Drevarr Mo| !Serafina E| !Silvyrs| !Mac Ottere| !Praetorian| !Reppalskay| !Chocolate | !Amalgam Ca| !Ms Elitia| !Upper D Am| !Adapted In| !Corrupted | !Hanbee Ele| !Mistwarden| !Komyni| !Hang L| !Kanta Kahn| !Arcana In | !Relexis| !Elemental | !Gandalffa |h
|Amitiels Revenge | 54,980| 954| | 72,390| 171| 11,202| 17,232| | 2,396| | | 398| | | | 1,746| | | | | |
|Moon In Pink | 65,984| | 770| 49,041| | 1,517| 16,535| | 1,528| | | 259| 967| | | | | | | | |
|Ms Elitia | 67,775| | 631| 55,042| | | 5,684| | 1,565| | | 1,350| | | | | | | | | |
|Drevarr Moonwillow | 97,296| 531| 269| 2,289| 178| 676| 1,303| | 154| | | | | | | | | | | | |
|Muxi W | 82,546| | 323| 7,338| | 2,775| 10,557| | | | | | 283| | | | | | | | |
|Newtype Clan | 94,319| | 622| 5,277| | | 9,777| | | | | 162| | | | 445| | | | | |
|Aezlenne | 22,155| | | 12,176| | 13,002| 65,640| | | | | 944| 714| | | 2,242| | | | | |
|Reppalskay | 37,827| 1,095| 388| 13,315| | 8,228| 57,991| | 307| | 197| 29| 488| | | 476| | |

In [88]:
json_datafile.close()